In [ ]:

from config.config import *
from pyspark.sql import functions as F

# Semantic model notes are in docs/powerbi_semantic_model.md and docs/powerbi_measures.md.
# Create DAX-ready summary tables with explicit grain.

dim_country = spark.read.format("delta").load(f"{gold_path}/dim_country")
dim_date = spark.read.format("delta").load(f"{gold_path}/dim_date")

# Total Arrivals (DAX-ready): sum of standardized_value for annual arrivals
fa = spark.read.format("delta").load(f"{gold_path}/fact_tourism_arrivals")
pbi_total_arrivals = (
    fa.filter(F.col("frequency") == "A")
      .groupBy("country_key", "country_name", "year")
      .agg(F.sum("standardized_value").alias("total_arrivals"))
      .withColumn("measure_name", F.lit("Total Arrivals"))
)
pbi_total_arrivals.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/pbi_total_arrivals")

# Average Occupancy Rate
occ = spark.read.format("delta").load(f"{gold_path}/fact_hotel_occupancy")
pbi_avg_occupancy = (
    occ.groupBy("country_key", "year", "month")
       .agg(F.avg("occupancy_rate").alias("avg_occupancy_rate"))
       .withColumn("measure_name", F.lit("Average Occupancy Rate"))
)
pbi_avg_occupancy.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/pbi_avg_occupancy")

print("pbi_total_arrivals:", pbi_total_arrivals.count(), "pbi_avg_occupancy:", pbi_avg_occupancy.count())
